# Comparison of Two Stocks Using Python
**PBL Activity 3 – Micro Project | Python for Data Science (BE05000231)**

A **universal, reusable** two-stock comparison tool. Upload the data for **Stock 1**, then the data for **Stock 2** (one file at a time, CSV or Excel) and the notebook automatically:

1. detects each company name from its file name (e.g. `RELIANCE.csv` → **RELIANCE**),
2. recognises the important columns even when they are named differently (`Date`/`date`/`DATE`, `Close`/`Closing Price`/`Price`, `Adj Close`, `Vol.`, …),
3. validates each file (required columns, date format, missing values, duplicates, invalid prices) and reports problems in plain language,
4. cleans each file (dates, duplicates, missing values, invalid rows, symbol renames),
5. detects and adjusts **stock splits / bonus issues** so returns are not distorted,
6. aligns both stocks on their **common trading dates**,
7. analyses and compares price, returns, volatility, moving averages, risk, correlation and statistical significance,
8. runs two **beyond-syllabus** techniques: Monte Carlo price simulation (Geometric Brownian Motion) and two-stock portfolio optimisation (Efficient Frontier),
9. prints a final comparison report and exports the cleaned data, summary tables and every figure.

Every graph title, legend, table column and output file uses the names of the two uploaded stocks – nothing is hard-coded.

**Supported files:** Kaggle *NIFTY-50 Stock Market Data*, Yahoo Finance downloads, Investing.com / NSE exports, or any daily file with at least a date column and a closing-price column.

### How to use
```text
1. Runtime → Run all
2. Step 3: click "Choose Files" and upload the Stock 1 file
3. Step 4: click "Choose Files" and upload the Stock 2 file
4. Everything else runs automatically → validation → cleaning → comparison → final report → download
```
To compare a different pair, simply **Run all** again and upload two new files. No code needs to be edited.

## Step 1 – Import libraries and set configuration
The settings below are optional – the notebook works without changing anything.

In [ ]:
import os, re, io, zipfile, warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from IPython.display import display

warnings.filterwarnings("ignore")
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
sns.set_theme(style="whitegrid")
plt.rcParams.update({"figure.figsize": (12, 5), "figure.dpi": 100, "axes.titleweight": "bold"})

# ------------------- CONFIGURATION (optional – no editing needed for a new pair of stocks) -------------------
START_DATE      = None      # e.g. "2015-01-01" to restrict the analysis window; None = full common period
END_DATE        = None      # e.g. "2020-12-31"
ADJUST_SPLITS   = True      # detect and back-adjust stock splits / bonus issues
SERIES_FILTER   = "EQ"      # keep only this NSE series if a 'Series' column exists (None = keep all)
TRADING_DAYS    = 252       # trading days per year, used to annualise
RISK_FREE_RATE  = "auto"    # "auto" = 6.5% (Indian 10-yr G-Sec) for ₹ stocks, 4.0% otherwise; or a number, e.g. 0.07
CURRENCY        = "auto"    # "auto" = ₹ when both files look like NSE/BSE data, otherwise no symbol; or e.g. "$"
ROLL_WINDOW     = 30        # rolling window (days) for volatility
SMA_SHORT, SMA_LONG = 50, 200
MC_SIMULATIONS  = 1000      # Monte Carlo paths
MC_DAYS         = 252       # Monte Carlo horizon (trading days)
RANDOM_SEED     = 42
MIN_ROWS        = 30        # a file needs at least this many rows with a valid date and price
MIN_COMMON_DAYS = 60        # the two files must share at least this many trading days
MAX_UPLOAD_ATTEMPTS = 3     # in Colab, an invalid file can be re-uploaded this many times
OUTPUT_DIR      = "stock_comparison_outputs"
# Fallback paths used only when NOT running in Google Colab (e.g. local Jupyter): [Stock 1 file, Stock 2 file]
LOCAL_FILES     = ["Stock_ACC.xlsx", "Stock_ADANIPORTS.xlsx"]
# -------------------------------------------------------------------------------------------------------------

try:
    from google.colab import files as colab_files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

class StockDataError(Exception):
    """A problem with an uploaded file – shown as a plain message instead of a long Python traceback."""

def _friendly_error(shell, etype, value, tb, tb_offset=None):
    print("\n" + "=" * 70)
    print("❌  DATA PROBLEM")
    print("=" * 70)
    print(value)
    print("\nPlease fix the file (or choose a different one) and run the notebook again (Runtime → Run all).")
    return None
try:
    get_ipython().set_custom_exc((StockDataError,), _friendly_error)   # stops "Run all" with a clear message
except NameError:
    pass

def safe_name(text):
    """Make a stock name safe for file names (spaces and symbols → underscores)."""
    return re.sub(r"[^A-Za-z0-9]+", "_", str(text)).strip("_") or "stock"

os.makedirs(OUTPUT_DIR, exist_ok=True)
for f in os.listdir(OUTPUT_DIR):                      # clear outputs from a previous pair
    os.remove(os.path.join(OUTPUT_DIR, f))
fig_counter = {"n": 0}
def save_fig(name):
    """Save the current figure with a running number so outputs stay in notebook order."""
    fig_counter["n"] += 1
    path = os.path.join(OUTPUT_DIR, f"fig{fig_counter['n']:02d}_{safe_name(name)}.png")
    plt.savefig(path, dpi=150, bbox_inches="tight")
    return path

print("Libraries loaded. pandas", pd.__version__, "| numpy", np.__version__,
      "| running in", "Google Colab" if IN_COLAB else "local Jupyter")

## Step 2 – Data-loading and validation functions
These functions make the notebook universal:

* **`read_any_file()`** – reads CSV (comma, semicolon or tab separated) and Excel files.
* **`standardise_columns()`** – maps common column-name variations to standard names (`Date`, `Open`, `High`, `Low`, `Close`, `Adj Close`, `Volume`, …). If there is no `Close` column, `Price`, `Last` or `Adj Close` is used as the closing price. The original data is kept unchanged in `stock1_raw` / `stock2_raw`.
* **`parse_dates()`** – handles ISO dates, day-first dates (`31/12/2020`), time zones and Excel date numbers.
* **`to_number()`** – converts text prices/volumes such as `"1,234.50"`, `"₹850"`, `"$120"` or `"2.5M"` to numbers.
* **`detect_stock_name()`** – takes the company name from the file name (`Stock_ACC.xlsx` → `ACC`, `TCS.NS.csv` → `TCS`, `HDFC_Bank.csv` → `HDFC Bank`), falling back to a `Symbol` column.
* **`validate_stock_data()`** – checks required columns, date format, missing values, duplicates and invalid prices.
* **`upload_stock()`** – shows one upload button, loads and validates the file, and prints a confirmation.

In [ ]:
COLUMN_ALIASES = {
    # date
    "date": "Date", "dates": "Date", "timestamp": "Date", "datetime": "Date", "trade date": "Date",
    "trading date": "Date", "day": "Date",
    # identifiers
    "symbol": "Symbol", "ticker": "Symbol", "stock symbol": "Symbol", "series": "Series",
    # prices
    "prev close": "Prev Close", "prevclose": "Prev Close", "previous close": "Prev Close", "prev close price": "Prev Close",
    "open": "Open", "open price": "Open", "opening price": "Open",
    "high": "High", "high price": "High", "day high": "High",
    "low": "Low", "low price": "Low", "day low": "Low",
    "last": "Last", "last price": "Last", "ltp": "Last", "last traded price": "Last",
    "close": "Close", "close price": "Close", "closing price": "Close", "closeprice": "Close", "close/last": "Close",
    "adj close": "Adj Close", "adjclose": "Adj Close", "adjusted close": "Adj Close",
    "adj close price": "Adj Close", "adjusted close price": "Adj Close", "adjusted closing price": "Adj Close",
    "vwap": "VWAP", "average price": "VWAP", "avg price": "VWAP",
    # activity
    "volume": "Volume", "vol": "Volume", "shares traded": "Volume", "total traded quantity": "Volume",
    "traded quantity": "Volume", "no of shares": "Volume",
    "turnover": "Turnover", "traded value": "Turnover", "total traded value": "Turnover",
    "trades": "Trades", "no of trades": "Trades", "number of trades": "Trades", "total trades": "Trades",
    "deliverable volume": "Deliverable Volume", "deliverable qty": "Deliverable Volume",
    "deliverable quantity": "Deliverable Volume",
    "%deliverble": "%Deliverable", "%deliverable": "%Deliverable", "% deliverable": "%Deliverable",
    "deliverable %": "%Deliverable", "% dly qt to traded qty": "%Deliverable",
}
CLOSE_FALLBACKS = ["Price", "Last", "Adj Close"]   # used as the closing price when no 'Close' column exists
PRICE_COLS   = ["Prev Close", "Open", "High", "Low", "Last", "Close", "VWAP", "Adj Close"]
NUMERIC_COLS = PRICE_COLS + ["Volume", "Turnover", "Trades", "Deliverable Volume", "%Deliverable"]
REQUIRED     = ["Date", "Close"]
SUPPORTED    = (".csv", ".xlsx", ".xls")

def read_any_file(name, raw_bytes):
    """Read a CSV or Excel file from bytes. Raises StockDataError with a clear message if it cannot be read."""
    lower = name.lower()
    if not lower.endswith(SUPPORTED):
        raise StockDataError(f"'{name}' is not a CSV or Excel file. Supported types: {', '.join(SUPPORTED)}")
    try:
        if lower.endswith(".csv"):
            for enc in ("utf-8-sig", "latin1"):
                try:
                    df = pd.read_csv(io.BytesIO(raw_bytes), encoding=enc)
                    if df.shape[1] == 1:                     # probably ';' or tab separated
                        df = pd.read_csv(io.BytesIO(raw_bytes), encoding=enc, sep=None, engine="python")
                    return df
                except UnicodeDecodeError:
                    continue
        return pd.read_excel(io.BytesIO(raw_bytes))
    except StockDataError:
        raise
    except ImportError as e:
        raise StockDataError(f"Cannot read '{name}': a library is missing ({e}). Try saving the file as .csv or .xlsx.")
    except Exception as e:
        raise StockDataError(f"Cannot read '{name}'. The file may be empty or damaged ({type(e).__name__}: {e}).")

def _column_key(c):
    k = str(c).strip().lower().replace("_", " ")
    k = re.sub(r"\(.*?\)", "", k)                       # drop units such as "(₹)" or "(Lacs)"
    k = k.replace("*", "").replace(".", "")
    return re.sub(r"\s+", " ", k).strip()

def standardise_columns(df):
    """Return a copy with standard column names, plus a list of notes about what was mapped."""
    notes, mapping, used = [], {}, set()
    for c in df.columns:
        target = COLUMN_ALIASES.get(_column_key(c))
        if _column_key(c) == "price":
            target = "Price"
        if target and target not in used:
            mapping[c] = target; used.add(target)
            if str(c).strip() != target:
                notes.append(f"column '{c}' → '{target}'")
        else:
            mapping[c] = str(c).strip()
    df = df.rename(columns=mapping)
    if "Close" not in df.columns:
        for fb in CLOSE_FALLBACKS:
            if fb in df.columns:
                df["Close"] = df[fb]
                notes.append(f"no 'Close' column – '{fb}' used as the closing price")
                break
    if "Date" not in df.columns:
        if isinstance(df.index, pd.DatetimeIndex):
            df = df.reset_index().rename(columns={df.index.name or "index": "Date"})
            notes.append("date taken from the index")
        elif len(df.columns) and parse_dates(df.iloc[:, 0])[0].notna().mean() > 0.8:
            notes.append(f"first column '{df.columns[0]}' recognised as the date")
            df = df.rename(columns={df.columns[0]: "Date"})
    return df, notes

def to_number(s):
    """Convert text such as '1,234.5', '₹850', '$120', '45%' or '2.5M' to numbers (invalid → NaN)."""
    if pd.api.types.is_numeric_dtype(s):
        return s.astype(float)
    t = s.astype(str).str.strip()
    t = t.str.replace(r"(?i)^rs\.?", "", regex=True).str.replace(r"[,\s₹$€£%]", "", regex=True)
    t = t.where(~t.str.lower().isin(["", "-", "null", "none", "nan", "na", "n/a"]))
    last = t.str[-1].str.upper()
    mult = last.map({"K": 1e3, "M": 1e6, "B": 1e9}).fillna(1.0)
    t = t.where(~last.isin(["K", "M", "B"]), t.str[:-1])
    return pd.to_numeric(t, errors="coerce") * mult

def parse_dates(s):
    """Parse a date column robustly. Returns (datetime Series, description of the detected format)."""
    if pd.api.types.is_datetime64_any_dtype(s):
        out = s.dt.tz_localize(None) if getattr(s.dt, "tz", None) is not None else s
        return out.dt.normalize(), "already a date type"
    if pd.api.types.is_numeric_dtype(s):
        med = s.median()
        if 20000 < med < 80000:
            return pd.to_datetime(s, unit="D", origin="1899-12-30", errors="coerce").dt.normalize(), "Excel date numbers"
        if med > 1e11:
            return pd.to_datetime(s, unit="ms", errors="coerce").dt.normalize(), "Unix timestamps (ms)"
        if med > 1e8:
            return pd.to_datetime(s, unit="s", errors="coerce").dt.normalize(), "Unix timestamps (s)"
    txt = s.astype(str).str.strip()
    txt = txt.str.replace(r"(?:\s*(?:[+-]\d{2}:?\d{2}|Z|UTC|GMT))$", "", regex=True)   # drop time-zone suffix
    parts = txt.str.extract(r"^(\d{1,2})[/\-. ](\d{1,2})[/\-. ](\d{2,4})")
    dayfirst, fmt = False, "year-first / text month"
    if parts[0].notna().mean() > 0.5:
        a, b = pd.to_numeric(parts[0], errors="coerce"), pd.to_numeric(parts[1], errors="coerce")
        if (a > 12).any() and not (b > 12).any():
            dayfirst, fmt = True, "day-first (DD/MM/YYYY)"
        elif (b > 12).any():
            fmt = "month-first (MM/DD/YYYY)"
        else:
            dayfirst, fmt = True, "ambiguous – assumed day-first (DD/MM/YYYY)"
    out = pd.to_datetime(txt, dayfirst=dayfirst, errors="coerce")
    if out.isna().mean() > 0.05 and int(pd.__version__.split(".")[0]) >= 2:
        alt = pd.to_datetime(txt, dayfirst=dayfirst, errors="coerce", format="mixed")
        if alt.isna().sum() < out.isna().sum():
            out = alt
    if getattr(out.dt, "tz", None) is not None:
        out = out.dt.tz_localize(None)
    return out.dt.normalize(), fmt

NAME_NOISE = {"stock", "stocks", "share", "shares", "data", "dataset", "historical", "history", "hist", "price",
              "prices", "daily", "nse", "bse", "eq", "equity", "yahoo", "finance", "quote", "quotes", "download",
              "downloaded", "file", "ohlc", "ohlcv", "final", "clean", "cleaned", "copy", "export", "full",
              "from", "to", "till", "and", "of", "the", "csv", "xlsx", "xls", "new", "old", "raw", "ns", "bo"}
MONTHS = r"(jan|feb|mar|apr|may|jun|jul|aug|sep|sept|oct|nov|dec)[a-z]*"

def detect_stock_name(file_name, df=None):
    """Company name from the file name (noise words and dates removed); fallback: Symbol column."""
    base = os.path.splitext(os.path.basename(file_name))[0]
    base = re.sub(r"\s*\(\d+\)$", "", base)                       # Colab adds ' (1)' to repeated uploads
    base = re.sub(r"(?i)\.(ns|bo)$", "", base)                    # Yahoo exchange suffix
    tokens = [t for t in re.split(r"[\s_\-\.,]+", base) if t]
    keep = [t for t in tokens
            if t.lower() not in NAME_NOISE
            and not re.fullmatch(r"\d+", t)
            and not re.fullmatch(rf"(?i)\d{{0,2}}{MONTHS}\d{{0,4}}", t)]
    if keep:
        return " ".join(keep), "file name"
    if df is not None and "Symbol" in df.columns and df["Symbol"].notna().any():
        return str(df["Symbol"].dropna().iloc[-1]).strip(), "Symbol column"
    return None, None

def load_stock_data(file_name, raw_bytes, slot):
    """Read one file and return a dict with the original data, a standardised copy and its stock name."""
    raw_df = read_any_file(file_name, raw_bytes)
    if raw_df.empty:
        raise StockDataError(f"'{file_name}' contains no rows.")
    df, notes = standardise_columns(raw_df.copy())               # the original raw_df is never modified
    missing = [c for c in REQUIRED if c not in df.columns]
    if missing:
        raise StockDataError(
            f"'{file_name}' is not a compatible stock file – required column(s) {missing} not found.\n"
            f"Columns found: {list(raw_df.columns)}\n"
            "The file needs a date column (e.g. Date / date / DATE) and a closing-price column "
            "(e.g. Close / Closing Price / Price / Adj Close).")
    df["Date"], date_fmt = parse_dates(df["Date"])
    for c in NUMERIC_COLS:
        if c in df.columns:
            df[c] = to_number(df[c])
    name, source = detect_stock_name(file_name, df)
    if not name:
        name, source = f"Stock {slot}", "default (name could not be detected)"
    return {"slot": slot, "file": file_name, "bytes": raw_bytes, "name": name, "name_source": source,
            "raw": raw_df, "df": df, "notes": notes, "date_format": date_fmt}

def validate_stock_data(info):
    """Check one standardised dataset; raise StockDataError if it is unusable, else return a checks dict."""
    df, name = info["df"], info["name"]
    valid_date = df["Date"].notna()
    valid_close = df["Close"].notna() & (df["Close"] > 0)
    usable = valid_date & valid_close
    checks = {
        "File": info["file"], "Name detected from": info["name_source"],
        "Rows": len(df), "Columns": df.shape[1],
        "Required columns (Date, Close)": "present",
        "Optional columns found": ", ".join(c for c in ["Open", "High", "Low", "Adj Close", "Volume"] if c in df.columns) or "none",
        "Date format detected": info["date_format"],
        "Invalid / unreadable dates": int((~valid_date).sum()),
        "Missing Close values": int(df["Close"].isna().sum()),
        "Zero / negative Close": int((df["Close"] <= 0).sum()),
        "High < Low rows": int((df["High"] < df["Low"]).sum()) if {"High", "Low"}.issubset(df.columns) else "n/a",
        "Exact duplicate rows": int(info["raw"].duplicated().sum()),
        "Duplicate dates": int(df.loc[valid_date, "Date"].duplicated().sum()),
        "Missing values (all columns)": int(df.isna().sum().sum()),
        "Usable rows (valid date + price)": int(usable.sum()),
    }
    if valid_date.sum() == 0:
        raise StockDataError(f"{name} ('{info['file']}'): none of the values in the date column could be read as dates.\n"
                             "Use a standard format such as YYYY-MM-DD or DD/MM/YYYY.")
    if valid_close.sum() == 0:
        raise StockDataError(f"{name} ('{info['file']}'): the closing-price column has no valid positive numbers.")
    if usable.sum() < MIN_ROWS:
        raise StockDataError(f"{name} ('{info['file']}'): only {int(usable.sum())} rows have both a valid date and a "
                             f"positive closing price – at least {MIN_ROWS} are needed for a meaningful comparison.")
    gaps = df.loc[usable, "Date"].sort_values().diff().dt.days.dropna()
    info["warnings"] = []
    if len(gaps) and gaps.median() > 5:
        info["warnings"].append(f"dates are on average {gaps.median():.0f} days apart – this looks like weekly/monthly "
                                "data; annualised figures assume DAILY data")
    if (~valid_date).mean() > 0.10:
        info["warnings"].append(f"{(~valid_date).mean():.0%} of dates could not be read and will be dropped")
    info["date_min"], info["date_max"] = df.loc[usable, "Date"].min(), df.loc[usable, "Date"].max()
    info["checks"] = checks
    return checks

def _receive_file(slot):
    """Show one upload button (Colab) or use LOCAL_FILES (local Jupyter). Returns (file name, bytes)."""
    if IN_COLAB:
        print(f"⬆  Choose the data file for STOCK {slot} (one CSV or Excel file)...")
        uploaded = colab_files.upload()
        if not uploaded:
            raise StockDataError(f"No file was selected for Stock {slot}.")
        names = list(uploaded.keys())
        if len(names) > 1:
            print(f"   More than one file selected – using only '{names[0]}' for Stock {slot}.")
        return names[0], uploaded[names[0]]
    if len(LOCAL_FILES) < slot:
        raise StockDataError(f"Not running in Colab and LOCAL_FILES has no entry for Stock {slot}.")
    path = LOCAL_FILES[slot - 1]
    if not os.path.exists(path):
        raise StockDataError(f"Not running in Colab and the local file '{path}' (Stock {slot}) was not found. "
                             "Edit LOCAL_FILES in Step 1 or run the notebook in Google Colab.")
    print(f"Not running in Colab – reading Stock {slot} from LOCAL_FILES: {path}")
    with open(path, "rb") as fh:
        return os.path.basename(path), fh.read()

def upload_stock(slot, other=None):
    """Upload, load and validate ONE stock file. In Colab, an invalid file can be replaced straight away."""
    for attempt in range(1, MAX_UPLOAD_ATTEMPTS + 1):
        try:
            file_name, raw_bytes = _receive_file(slot)
            info = load_stock_data(file_name, raw_bytes, slot)
            if other is not None and raw_bytes == other["bytes"]:
                raise StockDataError(f"'{file_name}' is identical to the Stock {other['slot']} file "
                                     f"('{other['file']}'). Please upload a DIFFERENT stock.")
            validate_stock_data(info)
            break
        except StockDataError as e:
            if not IN_COLAB or attempt == MAX_UPLOAD_ATTEMPTS:
                raise
            print(f"\n✘ {e}\n   → Please upload a valid file for Stock {slot} "
                  f"(attempt {attempt + 1} of {MAX_UPLOAD_ATTEMPTS}).\n")
    if other is not None and info["name"].strip().lower() == other["name"].strip().lower():
        new = f"{info['name']} (2)"
        print(f"Note: both files are named '{info['name']}' – Stock {slot} will be shown as '{new}'.")
        info["name"] = new
    print(f"\n✔ Stock {slot} uploaded successfully")
    print(f"  File       : {info['file']}  ({len(info['bytes'])/1024:,.1f} KB)")
    print(f"  Company    : {info['name']}   (detected from {info['name_source']})")
    print(f"  Rows       : {len(info['df']):,}")
    print(f"  Columns    : {info['raw'].shape[1]}")
    print(f"  Date Range : {info['date_min'].date()} to {info['date_max'].date()}")
    for n in info["notes"]:
        print(f"  • {n}")
    for w in info["warnings"]:
        print(f"  ⚠ {w}")
    return info

print("Loading and validation functions ready.")

## Step 3 – Upload Stock 1
Click **Choose Files** and select the data file for the **first** stock (e.g. `RELIANCE.csv`). The company name is taken from the file name.

In [ ]:
stock1 = upload_stock(1)
stock1_name = stock1["name"]
stock1_raw  = stock1["raw"]      # original data exactly as uploaded (never modified)
stock1_df   = stock1["df"]       # standardised copy used for the analysis

## Step 4 – Upload Stock 2
Click **Choose Files** and select the data file for the **second** stock (e.g. `TCS.csv`).

In [ ]:
stock2 = upload_stock(2, other=stock1)
stock2_name = stock2["name"]
stock2_raw  = stock2["raw"]
stock2_df   = stock2["df"]

print("\n" + "=" * 50)
print(f"Stock 1 = {stock1_name}")
print(f"Stock 2 = {stock2_name}")
print("=" * 50)

## Step 5 – Data validation report
Checks both datasets before analysis: required columns, date format, missing values, duplicates, invalid prices and whether both contain usable date and closing-price data. The currency label and risk-free rate are also set automatically.

In [ ]:
raw = {stock1_name: stock1_df, stock2_name: stock2_df}           # standardised data, keyed by stock name

validation_report = pd.DataFrame({stock1_name: stock1["checks"], stock2_name: stock2["checks"]})
display(validation_report)

for s, info in ((stock1_name, stock1), (stock2_name, stock2)):
    print(f"\nFirst rows of {s} (standardised columns):")
    display(info["df"].head())
print(f"\nColumn data types ({stock1_name}):")
print(stock1_df.dtypes)

# ---- currency label and risk-free rate (automatic) ----
def looks_indian(info):
    cols = set(info["df"].columns)
    return bool({"Series", "%Deliverable", "Deliverable Volume"} & cols) or \
           bool(re.search(r"(?i)(\.ns|\.bo|nse|bse|nifty|sensex)", info["file"]))
both_indian = looks_indian(stock1) and looks_indian(stock2)
CUR = ("₹" if both_indian else "") if CURRENCY == "auto" else (CURRENCY or "")
RF  = (0.065 if both_indian else 0.04) if RISK_FREE_RATE == "auto" else float(RISK_FREE_RATE)

def cu(label):
    """Add the currency symbol to a label when it is known, e.g. 'Price' → 'Price (₹)'."""
    return f"{label} ({CUR})" if CUR else label
def money(x):
    return f"{CUR}{x:,.2f}"

print(f"\nCurrency label : {CUR or '(none – prices shown in each file’s own currency)'}")
print(f"Risk-free rate : {RF:.2%} per year" + ("  (auto)" if RISK_FREE_RATE == "auto" else ""))
if not both_indian and CURRENCY == "auto":
    print("Note: returns, risk and statistics are scale-free, so stocks quoted in different currencies can still be compared;"
          " only the raw price charts are in each stock's own currency.")

## Step 6 – Clean each stock
Dates are parsed and sorted; exact and same-date duplicates are removed; prices are converted to numbers; rows with a missing or non-positive close, or with High < Low, are dropped; symbol renames (e.g. `MUNDRAPORT → ADANIPORTS`) are reported; a file holding several *different* stocks is rejected.

In [ ]:
def clean_stock_data(label, df):
    log = {"Stock": label, "Rows (raw)": len(df)}
    df = df.copy()
    if not pd.api.types.is_datetime64_any_dtype(df["Date"]):
        df["Date"], _ = parse_dates(df["Date"])
    log["Invalid dates"] = int(df["Date"].isna().sum())
    df = df.dropna(subset=["Date"])

    if SERIES_FILTER and "Series" in df.columns:
        before = len(df)
        df = df[df["Series"].astype(str).str.strip().str.upper() == SERIES_FILTER]
        log[f"Non-{SERIES_FILTER} rows"] = before - len(df)

    # One file must describe one company (a rename is allowed: date ranges must not overlap)
    log["Symbol history"] = "-"
    if "Symbol" in df.columns and df["Symbol"].nunique() > 1:
        ranges = df.groupby("Symbol")["Date"].agg(["min", "max"]).sort_values("min")
        if (ranges["min"].values[1:] <= ranges["max"].values[:-1]).any():
            raise StockDataError(f"{label}: the file contains several different stocks {list(ranges.index)}. "
                                 "Upload one stock per file.")
        log["Symbol history"] = " → ".join(map(str, ranges.index))

    num_cols = [c for c in NUMERIC_COLS if c in df.columns]
    for c in num_cols:
        if not pd.api.types.is_numeric_dtype(df[c]):
            df[c] = to_number(df[c])

    before = len(df); df = df.drop_duplicates()
    log["Exact duplicates"] = before - len(df)
    before = len(df); df = df.sort_values("Date").drop_duplicates(subset="Date", keep="last")
    log["Duplicate dates"] = before - len(df)

    before = len(df); df = df[df["Close"].notna() & (df["Close"] > 0)]
    log["Missing/zero Close"] = before - len(df)
    if {"High", "Low"}.issubset(df.columns):
        before = len(df); df = df[~(df["High"] < df["Low"])]
        log["High < Low"] = before - len(df)

    log["Missing values left"] = int(df[num_cols].isna().sum().sum())
    log["Rows (clean)"] = len(df)
    if len(df) < MIN_ROWS:
        raise StockDataError(f"{label}: only {len(df)} valid rows remain after cleaning (minimum {MIN_ROWS}).")
    log["From"] = df["Date"].min().date(); log["To"] = df["Date"].max().date()
    return df.set_index("Date"), log

clean, logs = {}, []
for label in (stock1_name, stock2_name):
    clean[label], lg = clean_stock_data(label, raw[label])
    logs.append(lg)

cleaning_report = pd.DataFrame(logs).set_index("Stock").T.fillna("-")
display(cleaning_report)
na_left = pd.DataFrame({s: clean[s].isna().sum() for s in clean}).fillna(0).astype(int)
na_left = na_left[(na_left > 0).any(axis=1)]
if len(na_left):
    print("Columns that still contain missing values (not used for price calculations, so kept as-is):")
    display(na_left)

## Step 7 – Detect and adjust stock splits / bonus issues
A split makes the quoted price fall overnight (e.g. a 5-for-1 split turns 837 into 167) although shareholders lose nothing. Unadjusted, this looks like an 80 % crash. The tool flags overnight moves that match a standard split or bonus ratio and back-adjusts all earlier prices (and volumes) so the series is continuous. If a file has a complete `Adj Close` column (e.g. Yahoo Finance), that is used instead.

In [ ]:
# price factor = new price / old price for common corporate actions
SPLIT_FACTORS = {"2:1 split / 1:1 bonus": 1/2, "3:1 split / 2:1 bonus": 1/3, "4:1 split / 3:1 bonus": 1/4,
                 "5:1 split / 4:1 bonus": 1/5, "10:1 split": 1/10, "3:2 bonus": 2/5,
                 "1:2 reverse split": 2, "1:5 reverse split": 5, "1:10 reverse split": 10}
ADJ_PRICE_COLS = ["Open", "High", "Low", "Close", "Last", "VWAP", "Prev Close"]

def detect_and_adjust(label, df, tol=0.12):
    df = df.copy()
    if "Adj Close" in df.columns and df["Adj Close"].notna().all() and (df["Adj Close"] > 0).all() \
            and not df["Adj Close"].equals(df["Close"]):
        ratio = df["Adj Close"] / df["Close"]
        for c in ADJ_PRICE_COLS:
            if c in df.columns: df[c] = df[c] * ratio
        print(f"{label}: used the existing 'Adj Close' column for adjustment.")
        return df, []
    events = []
    prev_close = df["Close"].shift(1)
    ratio = df["Close"] / prev_close
    for date, r in ratio[(ratio < 0.6) | (ratio > 1.67)].items():
        # a split shows up at the open; a genuine crash develops during the day
        if "Open" in df.columns and pd.notna(df.at[date, "Open"]) and \
                abs((df.at[date, "Open"] / prev_close.at[date]) / r - 1) > 0.15:
            continue
        name, f = min(SPLIT_FACTORS.items(), key=lambda kv: abs(r / kv[1] - 1))
        if abs(r / f - 1) <= tol:
            events.append({"Stock": label, "Date": date.date(), "Close before": prev_close.at[date],
                           "Close on day": df.at[date, "Close"], "Observed ratio": round(r, 4),
                           "Matched action": name, "Adjustment factor": round(f, 4)})
            before = df.index < date
            for c in ADJ_PRICE_COLS:
                if c in df.columns: df.loc[before, c] *= f
            for c in ["Volume", "Deliverable Volume"]:
                if c in df.columns: df.loc[before, c] /= f
            if "Prev Close" in df.columns:          # the split day's Prev Close is also pre-split
                df.at[date, "Prev Close"] *= f
    return df, events

adj, all_events = {}, []
for label in (stock1_name, stock2_name):
    if ADJUST_SPLITS:
        adj[label], ev = detect_and_adjust(label, clean[label]); all_events += ev
    else:
        adj[label] = clean[label].copy()

if all_events:
    print("Corporate actions detected and adjusted:")
    display(pd.DataFrame(all_events))
else:
    print("No splits or bonus issues detected.")

for label in dict.fromkeys(e["Stock"] for e in all_events):
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(clean[label]["Close"], label="Raw close (as in file)", alpha=.6)
    ax.plot(adj[label]["Close"], label="Split-adjusted close", lw=1.4)
    first = True
    for e in all_events:
        if e["Stock"] == label:
            ax.axvline(pd.Timestamp(e["Date"]), color="red", ls="--", lw=1, label="Detected split" if first else None)
            first = False
    ax.set_title(f"{label}: raw vs split-adjusted closing price"); ax.set_ylabel(cu("Price")); ax.legend()
    save_fig(f"split_adjustment_{label}"); plt.show()

## Step 8 – Align both stocks on common trading dates
Both series are joined on dates present in **both** files (inner join, i.e. the equivalent of `pd.merge(stock1_df, stock2_df, on="Date", how="inner")`), and the optional `START_DATE` / `END_DATE` window is applied, so every comparison uses exactly the same days.

In [ ]:
def prepare_comparison_data(df1, df2, name1, name2):
    """Inner-join two cleaned stocks on their common dates. Returns (merged_df, common columns)."""
    common = [c for c in ["Open", "High", "Low", "Close", "VWAP", "Volume", "Turnover", "Deliverable Volume", "%Deliverable"]
              if c in df1.columns and c in df2.columns]
    merged = pd.concat({name1: df1[common], name2: df2[common]}, axis=1, join="inner").sort_index()
    if START_DATE: merged = merged[merged.index >= pd.Timestamp(START_DATE)]
    if END_DATE:   merged = merged[merged.index <= pd.Timestamp(END_DATE)]
    if len(merged) < MIN_COMMON_DAYS:
        raise StockDataError(
            f"Only {len(merged)} common trading days – the two files do not overlap enough to compare "
            f"(minimum {MIN_COMMON_DAYS}).\n"
            f"{name1}: {df1.index.min().date()} to {df1.index.max().date()}\n"
            f"{name2}: {df2.index.min().date()} to {df2.index.max().date()}"
            + (f"\nAnalysis window: {START_DATE} to {END_DATE}" if START_DATE or END_DATE else ""))
    return merged, common

only_1 = adj[stock1_name].index.difference(adj[stock2_name].index)
only_2 = adj[stock2_name].index.difference(adj[stock1_name].index)
merged_df, common_cols = prepare_comparison_data(adj[stock1_name], adj[stock2_name], stock1_name, stock2_name)

close = merged_df.xs("Close", axis=1, level=1)
years = (close.index[-1] - close.index[0]).days / 365.25
print(f"Dates only in {stock1_name}: {len(only_1)} | only in {stock2_name}: {len(only_2)}")
print(f"Common trading days used: {len(merged_df):,}  ({close.index[0].date()} → {close.index[-1].date()}, {years:.1f} years)")
display(close.head()); display(close.tail())

## Step 9 – Descriptive statistics of the closing price

In [ ]:
def describe_prices(s):
    return pd.Series({"Count": s.count(), cu("Mean"): s.mean(), cu("Median"): s.median(),
                      cu("Mode, rounded") if CUR else "Mode (rounded)": s.round(0).mode().iloc[0],
                      cu("Minimum"): s.min(), cu("Maximum"): s.max(), cu("Range"): s.max() - s.min(),
                      cu("Std. Deviation"): s.std(), "Variance": s.var(),
                      "Coeff. of Variation (%)": s.std() / s.mean() * 100,
                      cu("Q1"): s.quantile(.25), cu("Q3"): s.quantile(.75),
                      "Skewness": s.skew(), "Kurtosis": s.kurt()})

price_stats = pd.DataFrame({s: describe_prices(close[s]) for s in close})
display(price_stats)
print("Note: a higher price level does not mean a better investment – Step 11 compares returns, which are scale-free.")

## Step 10 – Price trend: actual and normalised

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 9), sharex=True)
for s in close:
    axes[0].plot(close[s], label=s, lw=1.2)
axes[0].set_title(f"{stock1_name} vs {stock2_name} Closing Price"); axes[0].set_ylabel(cu("Price")); axes[0].legend()

normalised = close / close.iloc[0] * 100
for s in normalised:
    axes[1].plot(normalised[s], label=s, lw=1.2)
axes[1].axhline(100, color="grey", ls="--", lw=.8)
axes[1].set_title(f"{stock1_name} vs {stock2_name}: growth of {CUR}100 invested on the first common day")
axes[1].set_ylabel(cu("Value")); axes[1].legend()
plt.tight_layout(); save_fig("price_trend"); plt.show()

for s in normalised:
    print(f"{CUR}100 invested in {s} grew to {money(normalised[s].iloc[-1])}")

## Step 11 – Returns analysis
Daily % return, cumulative return, total return over the period, CAGR (compound annual growth rate) and calendar-year returns.

In [ ]:
def calculate_returns(prices):
    """Simple and log daily returns of a price DataFrame (one column per stock)."""
    return prices.pct_change().dropna(), np.log(prices / prices.shift(1)).dropna()

returns, log_returns = calculate_returns(close)

ret_table = pd.DataFrame({s: {
    cu("Initial price"): close[s].iloc[0], cu("Final price"): close[s].iloc[-1],
    "Total return (%)": (close[s].iloc[-1] / close[s].iloc[0] - 1) * 100,
    "CAGR (%)": ((close[s].iloc[-1] / close[s].iloc[0]) ** (1 / years) - 1) * 100,
    "Mean daily return (%)": returns[s].mean() * 100,
    "Best day (%)": returns[s].max() * 100, "Best day date": returns[s].idxmax().date(),
    "Worst day (%)": returns[s].min() * 100, "Worst day date": returns[s].idxmin().date(),
    "Positive days (%)": (returns[s] > 0).mean() * 100} for s in close})
display(ret_table)

# Daily returns side by side and cumulative returns
fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)
for i, s in enumerate(returns):
    axes[0].plot(returns[s] * 100, lw=.6, alpha=.7, label=s, color=f"C{i}")
axes[0].set_title(f"{stock1_name} Daily Returns vs {stock2_name} Daily Returns"); axes[0].set_ylabel("Daily return (%)"); axes[0].legend()
cumulative = ((1 + returns).cumprod() - 1) * 100
for i, s in enumerate(cumulative):
    axes[1].plot(cumulative[s], lw=1.2, label=s, color=f"C{i}")
axes[1].axhline(0, color="grey", ls="--", lw=.8)
axes[1].set_title(f"{stock1_name} vs {stock2_name} Cumulative Returns"); axes[1].set_ylabel("Cumulative return (%)"); axes[1].legend()
plt.tight_layout(); save_fig("daily_and_cumulative_returns"); plt.show()

year_end = close.groupby(close.index.year).last()
year_start_ref = pd.concat([close.iloc[[0]].set_axis([year_end.index[0] - 1]), year_end.iloc[:-1]])
yearly = (year_end.values / year_start_ref.values - 1) * 100
yearly = pd.DataFrame(yearly, index=year_end.index, columns=close.columns)
ax = yearly.plot(kind="bar", figsize=(13, 5), width=.8)
ax.axhline(0, color="black", lw=.8)
ax.set_title(f"{stock1_name} vs {stock2_name}: calendar-year return (%) – first and last years may be partial")
ax.set_ylabel("Return (%)"); ax.set_xlabel("Year")
plt.xticks(rotation=0); plt.tight_layout(); save_fig("yearly_returns"); plt.show()
display(yearly)
print(f"{stock1_name} beat {stock2_name} in {(yearly[stock1_name] > yearly[stock2_name]).sum()} of {len(yearly)} calendar years.")

## Step 12 – Volatility and return distribution

In [ ]:
vol_table = pd.DataFrame({s: {
    "Daily volatility (%)": returns[s].std() * 100,
    "Annualised volatility (%)": returns[s].std() * np.sqrt(TRADING_DAYS) * 100,
    "Return skewness": returns[s].skew(), "Return kurtosis (excess)": returns[s].kurt()} for s in returns})
display(vol_table)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
x = np.linspace(-10, 10, 400)
for i, s in enumerate(returns):
    sns.histplot(returns[s] * 100, bins=150, stat="density", alpha=.4, label=s, ax=axes[0], color=f"C{i}")
    axes[0].plot(x, stats.norm.pdf(x, returns[s].mean() * 100, returns[s].std() * 100), lw=1.4, color=f"C{i}")
axes[0].set_xlim(-10, 10); axes[0].set_title(f"{stock1_name} vs {stock2_name}: distribution of daily returns (lines = normal fit)")
axes[0].set_xlabel("Daily return (%)"); axes[0].legend()

long_ret = (returns * 100).melt(var_name="Stock", value_name="Daily return (%)")
sns.boxplot(data=long_ret, x="Stock", y="Daily return (%)", ax=axes[1], fliersize=2)
axes[1].set_title(f"Box plot of daily returns: {stock1_name} vs {stock2_name}")
plt.tight_layout(); save_fig("return_distribution"); plt.show()

rolling_vol = returns.rolling(ROLL_WINDOW).std() * np.sqrt(TRADING_DAYS) * 100
rolling_vol.plot(figsize=(13, 4.5), lw=1)
plt.title(f"{stock1_name} vs {stock2_name} Volatility – {ROLL_WINDOW}-day rolling annualised (%)"); plt.ylabel("Volatility (%)")
plt.tight_layout(); save_fig("rolling_volatility"); plt.show()
print(f"{stock1_name} was the more volatile stock on {(rolling_vol[stock1_name] > rolling_vol[stock2_name]).mean()*100:.1f}% of days.")

## Step 13 – Moving averages and trend signals
Simple moving averages (50-day and 200-day). A *golden cross* (50-day crosses above 200-day) is a classic bullish signal; a *death cross* is bearish.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 9), sharex=True)
cross_rows = []
for ax, s in zip(axes, close):
    sma_s, sma_l = close[s].rolling(SMA_SHORT).mean(), close[s].rolling(SMA_LONG).mean()
    ax.plot(close[s], label="Close", lw=.9, alpha=.7)
    ax.plot(sma_s, label=f"SMA {SMA_SHORT}", lw=1.2); ax.plot(sma_l, label=f"SMA {SMA_LONG}", lw=1.2)
    above = (sma_s > sma_l)[sma_l.notna()].astype(int)
    change = above.diff()
    golden, death = change[change == 1].index, change[change == -1].index
    ax.scatter(golden, sma_s[golden], marker="^", color="green", s=70, zorder=5, label="Golden cross")
    ax.scatter(death, sma_s[death], marker="v", color="red", s=70, zorder=5, label="Death cross")
    ax.set_title(f"{s}: price with moving averages"); ax.set_ylabel(cu("Price")); ax.legend(loc="upper left", fontsize=8)
    has_long = sma_l.notna().any()
    cross_rows.append({"Stock": s, "Golden crosses": len(golden), "Death crosses": len(death),
                       f"Days above SMA {SMA_LONG} (%)": (close[s] > sma_l)[sma_l.notna()].mean() * 100 if has_long else np.nan,
                       "Trend on last day": ("Bullish (SMA50 > SMA200)" if sma_s.iloc[-1] > sma_l.iloc[-1] else "Bearish (SMA50 < SMA200)")
                                            if has_long else f"n/a (fewer than {SMA_LONG} days)"})
plt.tight_layout(); save_fig("moving_averages"); plt.show()
ma_table = pd.DataFrame(cross_rows).set_index("Stock")
display(ma_table)

## Step 14 – Risk analysis: drawdown, Sharpe, Sortino and Value at Risk

In [ ]:
def max_drawdown(series):
    dd = series / series.cummax() - 1
    trough = dd.idxmin(); peak_date = series[:trough].idxmax()
    after = series[trough:]
    rec = after[after >= series[peak_date]]
    return dd, dd.min(), peak_date, trough, (rec.index[0] if len(rec) else None)

rf_daily = (1 + RF) ** (1 / TRADING_DAYS) - 1
risk_rows, drawdowns = {}, {}
for s in returns:
    r = returns[s]
    dd, mdd, pk, tr, rec = max_drawdown(close[s]); drawdowns[s] = dd * 100
    cagr = (close[s].iloc[-1] / close[s].iloc[0]) ** (1 / years) - 1
    ann_vol = r.std() * np.sqrt(TRADING_DAYS)
    downside = np.sqrt((np.minimum(r - rf_daily, 0) ** 2).mean()) * np.sqrt(TRADING_DAYS)
    var_hist = np.percentile(r, 5)
    risk_rows[s] = {
        "CAGR (%)": cagr * 100, "Annualised volatility (%)": ann_vol * 100,
        "Sharpe ratio": (r.mean() - rf_daily) / r.std() * np.sqrt(TRADING_DAYS),
        "Sortino ratio": (r.mean() - rf_daily) * TRADING_DAYS / downside if downside > 0 else np.nan,
        "Max drawdown (%)": mdd * 100, "Drawdown peak": pk.date(), "Drawdown trough": tr.date(),
        "Recovered on": rec.date() if rec is not None else "Not yet recovered",
        "Calmar ratio": cagr / abs(mdd) if mdd < 0 else np.nan,
        "VaR 95% – historical (%)": var_hist * 100,
        "VaR 95% – parametric (%)": stats.norm.ppf(.05, r.mean(), r.std()) * 100,
        "CVaR 95% (%)": r[r <= var_hist].mean() * 100}
risk_table = pd.DataFrame(risk_rows)
display(risk_table)
print(f"Risk-free rate used: {RF:.2%} per year.")
print("VaR: on 95% of days the one-day loss should not exceed |VaR|. CVaR: the average loss on the worst 5% of days.")

dd_df = pd.DataFrame(drawdowns)
ax = dd_df.plot(figsize=(13, 4.5), lw=1)
for i, s in enumerate(dd_df): ax.fill_between(dd_df.index, dd_df[s], 0, alpha=.12, color=f"C{i}")
ax.set_title(f"{stock1_name} vs {stock2_name}: drawdown from previous peak (%)"); ax.set_ylabel("Drawdown (%)")
plt.tight_layout(); save_fig("drawdown"); plt.show()

ax = risk_table.loc[["CAGR (%)", "Annualised volatility (%)", "Max drawdown (%)"]].astype(float).plot(kind="bar", figsize=(10, 4.5))
ax.axhline(0, color="black", lw=.8); ax.set_title(f"Growth vs risk: {stock1_name} vs {stock2_name}"); ax.set_ylabel("%"); plt.xticks(rotation=0)
plt.tight_layout(); save_fig("return_vs_risk"); plt.show()

## Step 15 – Correlation and regression between the two stocks

In [ ]:
pearson_r, pearson_p = stats.pearsonr(returns[stock1_name], returns[stock2_name])
spearman_r, _ = stats.spearmanr(returns[stock1_name], returns[stock2_name])
reg = stats.linregress(returns[stock1_name], returns[stock2_name])
price_corr = close[stock1_name].corr(close[stock2_name])
print(f"Pearson correlation of daily returns : {pearson_r:.4f} (p = {pearson_p:.3g})")
print(f"Spearman rank correlation            : {spearman_r:.4f}")
print(f"Price-level correlation              : {price_corr:.4f}  (driven by long-term trends, less meaningful)")
print(f"Regression: {stock2_name} return = {reg.intercept*100:.4f}% + {reg.slope:.4f} × {stock1_name} return   (R² = {reg.rvalue**2:.4f})")

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
axes[0].scatter(returns[stock1_name] * 100, returns[stock2_name] * 100, s=6, alpha=.35)
xs = np.linspace(returns[stock1_name].min(), returns[stock1_name].max(), 50)
axes[0].plot(xs * 100, (reg.intercept + reg.slope * xs) * 100, color="red", lw=1.5, label=f"Regression line, slope = {reg.slope:.2f}")
axes[0].set_xlabel(f"{stock1_name} daily return (%)"); axes[0].set_ylabel(f"{stock2_name} daily return (%)")
axes[0].set_title(f"{stock1_name} vs {stock2_name} daily returns (r = {pearson_r:.2f})"); axes[0].legend()

heat = pd.DataFrame({f"{s} return": returns[s] for s in returns})
if "Volume" in common_cols:
    for s in close: heat[f"{s} volume"] = merged_df[(s, "Volume")].reindex(returns.index)
sns.heatmap(heat.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=axes[1], square=True)
axes[1].set_title("Correlation matrix")
plt.tight_layout(); save_fig("correlation"); plt.show()

corr_window = 90
rolling_corr = returns[stock1_name].rolling(corr_window).corr(returns[stock2_name])
rolling_corr.plot(figsize=(13, 4), lw=1, color="purple", label="Rolling correlation")
plt.axhline(pearson_r, ls="--", color="grey", label=f"Full-period r = {pearson_r:.2f}")
plt.title(f"{stock1_name} vs {stock2_name}: {corr_window}-day rolling correlation of daily returns"); plt.legend()
plt.tight_layout(); save_fig("rolling_correlation"); plt.show()

## Step 16 – Hypothesis testing (α = 0.05)
1. **Paired t-test** – are the mean daily returns different? (same dates, so the samples are paired)
2. **Levene test** – are the return variances (risk) different?
3. **Jarque–Bera test** – are daily returns normally distributed?

In [ ]:
ALPHA = 0.05
def decide(p): return "Reject H₀" if p < ALPHA else "Fail to reject H₀"

t_stat, t_p = stats.ttest_rel(returns[stock1_name], returns[stock2_name])
lev_stat, lev_p = stats.levene(returns[stock1_name], returns[stock2_name])
tests = [
    {"Test": "Paired t-test", "H₀": f"Mean daily returns of {stock1_name} and {stock2_name} are equal", "Statistic": t_stat, "p-value": t_p, "Decision": decide(t_p)},
    {"Test": "Levene test", "H₀": f"Return variances of {stock1_name} and {stock2_name} are equal", "Statistic": lev_stat, "p-value": lev_p, "Decision": decide(lev_p)},
]
for s in returns:
    jb, jb_p = stats.jarque_bera(returns[s])
    tests.append({"Test": f"Jarque–Bera ({s})", "H₀": "Daily returns are normally distributed", "Statistic": jb, "p-value": jb_p, "Decision": decide(jb_p)})
test_table = pd.DataFrame(tests).set_index("Test")
display(test_table)

riskier = stock1_name if returns[stock1_name].var() > returns[stock2_name].var() else stock2_name
normal = [s for s in returns if stats.jarque_bera(returns[s])[1] >= ALPHA]
test_conclusions = [
    f"Mean daily returns: {'significantly different' if t_p < ALPHA else 'no statistically significant difference'} (p = {t_p:.4f}).",
    f"Risk: {'variances differ significantly – ' + riskier + ' is riskier' if lev_p < ALPHA else 'variances are not significantly different'} (p = {lev_p:.4g}).",
    f"Normality: {'returns of ' + ', '.join(normal) + ' look normal' if normal else 'neither stock has normally distributed returns (fat tails)'}; "
    "normal-based risk measures may understate extreme losses."]
print("Conclusions:")
for c in test_conclusions: print(" •", c)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, s in zip(axes, returns):
    stats.probplot(returns[s], dist="norm", plot=ax); ax.set_title(f"Q–Q plot of daily returns: {s}")
plt.tight_layout(); save_fig("qq_plots"); plt.show()

## Step 17 – Trading activity (volume and delivery)
Runs only for columns present in both files.

In [ ]:
def month_end(df, how):
    """Resample to month ('M') or quarter ('Q') end; works on old and new pandas versions."""
    try:
        return df.resample(how + "E").mean()
    except ValueError:
        return df.resample(how).mean()

activity = {s: {} for s in close}
if "Volume" in common_cols:
    vol = merged_df.xs("Volume", axis=1, level=1)
    if vol.notna().any().all() and (vol > 0).any().all():
        ax = month_end(vol, "M").plot(figsize=(13, 4.5), lw=1.1, logy=True)
        ax.set_title(f"{stock1_name} vs {stock2_name}: average daily traded volume per month (log scale, split-adjusted)")
        ax.set_ylabel("Shares")
        plt.tight_layout(); save_fig("volume"); plt.show()
    price_col = "VWAP" if "VWAP" in common_cols else "Close"
    value_unit, value_div = (f"{CUR} crore", 1e7) if CUR == "₹" else (cu("millions"), 1e6)
    for s in vol:
        activity[s]["Avg daily volume (shares)"] = vol[s].mean()
        activity[s][f"Avg daily traded value ({value_unit})"] = (merged_df[(s, "Volume")] * merged_df[(s, price_col)]).mean() / value_div
if "%Deliverable" in common_cols:
    deliv = merged_df.xs("%Deliverable", axis=1, level=1)
    if deliv.max().max() <= 1.5: deliv = deliv * 100            # stored as a fraction in the Kaggle files
    month_end(deliv, "Q").plot(figsize=(13, 4), lw=1.2)
    plt.title(f"{stock1_name} vs {stock2_name}: average delivery percentage per quarter"); plt.ylabel("% of traded volume delivered")
    plt.tight_layout(); save_fig("delivery_pct"); plt.show()
    for s in deliv: activity[s]["Avg delivery (%)"] = deliv[s].mean()
activity_table = pd.DataFrame(activity).dropna(how="all")
if activity_table.size:
    display(activity_table)
    if "%Deliverable" in common_cols:
        print("A higher delivery % means more buyers took the shares into their accounts rather than trading intraday.")
else:
    print("Volume/delivery columns not available in both files – step skipped.")

## Step 18 – Beyond syllabus (1): Monte Carlo simulation with Geometric Brownian Motion
Future prices are simulated many times using each stock's historical drift (μ) and volatility (σ) of daily log returns:

$$S_{t+1} = S_t \cdot \exp\left(\left(\mu - \tfrac{1}{2}\sigma^2\right) + \sigma Z_t\right), \qquad Z_t \sim N(0,1)$$

The spread of simulated outcomes shows the range of plausible prices one year ahead under this model. It is a probabilistic risk tool, **not** a price forecast.

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)
mc_rows = {}
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for i, s in enumerate(close):
    mu, sigma, s0 = log_returns[s].mean(), log_returns[s].std(), close[s].iloc[-1]
    z = rng.standard_normal((MC_DAYS, MC_SIMULATIONS))
    paths = s0 * np.exp(np.cumsum((mu - 0.5 * sigma**2) + sigma * z, axis=0))
    paths = np.vstack([np.full(MC_SIMULATIONS, s0), paths])
    final = paths[-1]
    axes[0, i].plot(paths[:, :100], lw=.5, alpha=.5)
    axes[0, i].plot(np.median(paths, axis=1), color="black", lw=2, label="Median path")
    axes[0, i].set_title(f"{s}: 100 of {MC_SIMULATIONS} simulated paths")
    axes[0, i].set_xlabel("Trading days ahead"); axes[0, i].set_ylabel(cu("Price")); axes[0, i].legend()
    sns.histplot(final, bins=60, ax=axes[1, i], color=f"C{i}")
    axes[1, i].axvline(np.percentile(final, 5), color="black", ls=":", lw=1.2, label="5th / 95th percentile")
    axes[1, i].axvline(np.percentile(final, 95), color="black", ls=":", lw=1.2)
    axes[1, i].axvline(s0, color="red", lw=1.5, label=f"Last price {CUR}{s0:,.0f}")
    axes[1, i].set_title(f"{s}: simulated price after {MC_DAYS} days"); axes[1, i].set_xlabel(cu("Price")); axes[1, i].legend()
    mc_rows[s] = {cu("Last price"): s0, "Daily drift μ": mu, "Daily volatility σ": sigma,
                  cu("Mean simulated price"): final.mean(), cu("Median simulated price"): np.median(final),
                  cu("5th percentile"): np.percentile(final, 5), cu("95th percentile"): np.percentile(final, 95),
                  "Probability of loss (%)": (final < s0).mean() * 100,
                  "Probability of >20% gain (%)": (final > 1.2 * s0).mean() * 100}
fig.suptitle(f"Monte Carlo simulation: {stock1_name} vs {stock2_name}", fontweight="bold")
plt.tight_layout(); save_fig("monte_carlo"); plt.show()
mc_table = pd.DataFrame(mc_rows); display(mc_table)

## Step 19 – Beyond syllabus (2): Portfolio optimisation – Efficient Frontier (Modern Portfolio Theory)
Every split of money between the two stocks (0–100 % in 1 % steps) is evaluated. Because the stocks are not perfectly correlated, mixing them can reduce risk. The tool finds the **minimum-variance** and **maximum-Sharpe** portfolios.

In [ ]:
mean_ann = returns.mean() * TRADING_DAYS
cov_ann = returns.cov() * TRADING_DAYS
w = np.linspace(0, 1, 101)
weights = np.column_stack([w, 1 - w])                   # column 0 = Stock 1, column 1 = Stock 2
port_ret = weights @ mean_ann[[stock1_name, stock2_name]].values
port_vol = np.sqrt(np.einsum("ij,jk,ik->i", weights, cov_ann.loc[[stock1_name, stock2_name], [stock1_name, stock2_name]].values, weights))
port_sharpe = (port_ret - RF) / port_vol
i_min, i_max = port_vol.argmin(), port_sharpe.argmax()

plt.figure(figsize=(10, 6))
sc = plt.scatter(port_vol * 100, port_ret * 100, c=port_sharpe, cmap="viridis", s=18)
plt.colorbar(sc, label="Sharpe ratio")
for idx, lbl, m in [(100, f"100% {stock1_name}", "s"), (0, f"100% {stock2_name}", "D"),
                    (i_min, "Minimum variance", "*"), (i_max, "Maximum Sharpe", "P")]:
    plt.scatter(port_vol[idx] * 100, port_ret[idx] * 100, marker=m, s=200, edgecolor="black", label=lbl, zorder=5)
plt.xlabel("Annualised volatility (%)"); plt.ylabel("Expected annual return (%)")
plt.title(f"Efficient frontier: {stock1_name} + {stock2_name}"); plt.legend()
plt.tight_layout(); save_fig("efficient_frontier"); plt.show()

portfolio_table = pd.DataFrame({
    lbl: {f"Weight {stock1_name} (%)": weights[i, 0] * 100, f"Weight {stock2_name} (%)": weights[i, 1] * 100,
          "Expected return (%)": port_ret[i] * 100, "Volatility (%)": port_vol[i] * 100, "Sharpe ratio": port_sharpe[i]}
    for lbl, i in [(f"100% {stock1_name}", 100), (f"100% {stock2_name}", 0), ("50 / 50", 50),
                   ("Minimum variance", i_min), ("Maximum Sharpe", i_max)]})
display(portfolio_table)
print(f"Diversification benefit: the minimum-variance mix has {port_vol[i_min]*100:.2f}% volatility vs "
      f"{min(port_vol[0], port_vol[100])*100:.2f}% for the less risky single stock.")
if i_max in (0, 100):
    print("The maximum-Sharpe point is a corner (100% in one stock): the other stock adds no risk-adjusted benefit in this period.")
print(f"Expected return here is the arithmetic mean return × {TRADING_DAYS}, the standard input for portfolio theory; it is higher than CAGR.")

## Step 20 – Final stock comparison and verdict
A clean summary of every section, using the names of the two uploaded stocks.

In [ ]:
summary = pd.DataFrame({s: {
    cu("Initial price"): ret_table.at[cu("Initial price"), s], cu("Final price"): ret_table.at[cu("Final price"), s],
    cu("Average price"): price_stats.at[cu("Mean"), s], cu("Highest price"): price_stats.at[cu("Maximum"), s],
    cu("Lowest price"): price_stats.at[cu("Minimum"), s],
    "Total return (%)": ret_table.at["Total return (%)", s], "CAGR (%)": ret_table.at["CAGR (%)", s],
    "Average daily return (%)": ret_table.at["Mean daily return (%)", s],
    "Maximum daily return (%)": ret_table.at["Best day (%)", s], "Minimum daily return (%)": ret_table.at["Worst day (%)", s],
    "Annualised volatility (%)": risk_table.at["Annualised volatility (%)", s],
    "Sharpe ratio": risk_table.at["Sharpe ratio", s], "Sortino ratio": risk_table.at["Sortino ratio", s],
    "Max drawdown (%)": risk_table.at["Max drawdown (%)", s],
    "VaR 95% – historical (%)": risk_table.at["VaR 95% – historical (%)", s],
    "MC probability of loss (%)": mc_table.at["Probability of loss (%)", s]} for s in close}).astype(float)

rules = {"Total return (%)": True, "CAGR (%)": True, "Annualised volatility (%)": False, "Sharpe ratio": True,
         "Sortino ratio": True, "Max drawdown (%)": True, "VaR 95% – historical (%)": True, "MC probability of loss (%)": False}
def winner(metric, higher_is_better):
    a, b = summary.at[metric, stock1_name], summary.at[metric, stock2_name]
    if pd.isna(a) or pd.isna(b): return "n/a"
    if np.isclose(a, b): return "Tie"
    return (stock1_name if a > b else stock2_name) if higher_is_better else (stock1_name if a < b else stock2_name)
summary["Better"] = [winner(m, rules[m]) if m in rules else "" for m in summary.index]

score = summary["Better"][summary["Better"].isin([stock1_name, stock2_name])].value_counts()
best_ret = summary.at["CAGR (%)", "Better"]; best_risk = summary.at["Annualised volatility (%)", "Better"]
best_adj = summary.at["Sharpe ratio", "Better"]
other = lambda s: stock2_name if s == stock1_name else stock1_name
corr_text = ("strong co-movement, limited diversification." if pearson_r > .7
             else "moderate co-movement, so combining them reduces risk." if pearson_r > .3
             else "weak co-movement, good diversification.")

verdict_lines = [
    f"Period analysed : {close.index[0].date()} → {close.index[-1].date()} ({len(close):,} common trading days, {years:.1f} years)",
    "Metrics won     : " + (", ".join(f"{k} {v}/{len(rules)}" for k, v in score.items()) or "none"),
    (f"Higher growth   : {best_ret} (CAGR {summary.at['CAGR (%)', best_ret]:.2f}% vs {summary.at['CAGR (%)', other(best_ret)]:.2f}%)"
     if best_ret in (stock1_name, stock2_name) else f"Higher growth   : {best_ret}"),
    (f"Lower risk      : {best_risk} (volatility {summary.at['Annualised volatility (%)', best_risk]:.2f}% vs "
     f"{summary.at['Annualised volatility (%)', other(best_risk)]:.2f}%)" if best_risk in (stock1_name, stock2_name) else f"Lower risk      : {best_risk}"),
    f"Better risk-adjusted return (Sharpe): {best_adj}",
    f"Returns correlation r = {pearson_r:.2f} → {corr_text}",
    f"Max-Sharpe mix  : {weights[i_max,0]*100:.0f}% {stock1_name} / {weights[i_max,1]*100:.0f}% {stock2_name}",
    f"Min-variance mix: {weights[i_min,0]*100:.0f}% {stock1_name} / {weights[i_min,1]*100:.0f}% {stock2_name}",
]

line = "=" * 64
print(line); print("STOCK COMPARISON".center(64)); print(line)
print(f"Stock 1: {stock1_name}   (file: {stock1['file']})")
print(f"Stock 2: {stock2_name}   (file: {stock2['file']})")
print(line)

print("\n1. KEY METRICS")
display(summary)

print("\n2. RETURNS");                         display(ret_table)
print("\n3. VOLATILITY");                      display(vol_table)
print("\n4. MOVING-AVERAGE TREND");            display(ma_table)
print("\n5. RISK");                            display(risk_table)
print("\n6. CORRELATION")
print(f"   Pearson r (daily returns) = {pearson_r:.4f}, Spearman = {spearman_r:.4f}, price-level r = {price_corr:.4f}")
print(f"   {stock2_name} return = {reg.intercept*100:.4f}% + {reg.slope:.4f} × {stock1_name} return (R² = {reg.rvalue**2:.4f})")
print("\n7. HYPOTHESIS TESTS");                display(test_table)
for c in test_conclusions: print("   •", c)
if activity_table.size:
    print("\n8. TRADING ACTIVITY");            display(activity_table)
print("\n9. MONTE CARLO SIMULATION");          display(mc_table)
print("\n10. PORTFOLIO (EFFICIENT FRONTIER)");  display(portfolio_table)

print("\n" + "=" * 24 + " VERDICT " + "=" * 24)
for l in verdict_lines: print(l)
print("Disclaimer: historical analysis for academic purposes only, not investment advice.")

ax = summary.loc[["Total return (%)", "CAGR (%)", "Annualised volatility (%)", "Max drawdown (%)"], [stock1_name, stock2_name]].plot(kind="barh", figsize=(10, 4.5))
ax.axvline(0, color="black", lw=.8); ax.set_title(f"Key metrics side by side: {stock1_name} vs {stock2_name}"); ax.set_xlabel("%")
plt.tight_layout(); save_fig("summary_metrics"); plt.show()

## Step 21 – Export results
Saves the cleaned and aligned data, all result tables (one Excel workbook), a text report and every figure, zips them (the zip is named after the two stocks), and downloads the zip in Colab.

In [ ]:
data_out = merged_df.copy(); data_out.columns = [f"{s} {c}" for s, c in data_out.columns]
data_out.to_csv(os.path.join(OUTPUT_DIR, "cleaned_aligned_data.csv"))
tables = {"Summary": summary, "Validation": validation_report.astype(str), "Cleaning": cleaning_report.astype(str),
          "Price stats": price_stats, "Returns": ret_table.astype(str), "Yearly returns": yearly,
          "Volatility": vol_table, "Moving averages": ma_table.astype(str), "Risk": risk_table.astype(str),
          "Hypothesis tests": test_table, "Monte Carlo": mc_table, "Portfolio": portfolio_table}
if activity_table.size: tables["Activity"] = activity_table
with pd.ExcelWriter(os.path.join(OUTPUT_DIR, "comparison_results.xlsx")) as xw:
    pd.DataFrame({"Item": ["Stock 1", "Stock 2", "Stock 1 file", "Stock 2 file", "Period", "Common trading days", "Risk-free rate"],
                  "Value": [stock1_name, stock2_name, stock1["file"], stock2["file"],
                            f"{close.index[0].date()} to {close.index[-1].date()}", len(close), f"{RF:.2%}"]}
                 ).to_excel(xw, sheet_name="Info", index=False)
    for sheet, t in tables.items(): t.to_excel(xw, sheet_name=sheet[:31])
    if all_events: pd.DataFrame(all_events).to_excel(xw, sheet_name="Splits adjusted", index=False)

with open(os.path.join(OUTPUT_DIR, "comparison_report.txt"), "w", encoding="utf-8") as fh:
    fh.write("STOCK COMPARISON\n")
    fh.write(f"Stock 1: {stock1_name} (file: {stock1['file']})\nStock 2: {stock2_name} (file: {stock2['file']})\n\n")
    fh.write("KEY METRICS\n" + summary.to_string(float_format=lambda v: f"{v:,.4f}") + "\n\n")
    fh.write("HYPOTHESIS TESTS\n" + "\n".join(" - " + c for c in test_conclusions) + "\n\n")
    fh.write("VERDICT\n" + "\n".join(verdict_lines) + "\n")
    fh.write("\nDisclaimer: historical analysis for academic purposes only, not investment advice.\n")

zip_name = f"comparison_{safe_name(stock1_name)}_vs_{safe_name(stock2_name)}.zip"
with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(os.listdir(OUTPUT_DIR)): z.write(os.path.join(OUTPUT_DIR, f), f)
print(f"Saved {len(os.listdir(OUTPUT_DIR))} files in '{OUTPUT_DIR}/' and zipped them as {zip_name}:")
for f in sorted(os.listdir(OUTPUT_DIR)): print("  -", f)
if IN_COLAB:
    colab_files.download(zip_name)